# v10: Evidence Window Coverage Deep Research Agent

v9 恢复到 38%，但 trace 显示所有 `get_document_window` 都从 `start=0` 读取，99% 以上窗口被截断，而文档平均长度超过 10 万字符。v10 保留 v9 的保守 Open Track 策略，重点优化工具读取覆盖率：在 `collect_evidence_snippets` 返回关键词命中 offset 后，额外打开这些 offset 附近的上下文窗口，让模型看到文档中段/后段的直接证据。

本 notebook 只定义方案和代码。本地不要执行、不要安装依赖、不要构建索引、不要启动或调用 vLLM。主轨迹写入 `./trace/v10_trace.jsonl`，Open Track 组件轨迹额外写入 `./trace/v10_open_track_trace.jsonl`。


## 1. 配置与导入

v10 继续复用现有 BM25 和 v3 工具注册，不修改 `agent/tools.py`。差异在 controller 层：扩大默认窗口，并基于 evidence snippet 的 `start` offset 追加读取非开头窗口。


In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_v3_research_tool_specs_and_registry, make_initial_search_queries
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v10_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v10_eval_results.jsonl")
TRACE_DIR = project_root / "trace"
TRACE_PATH = str(TRACE_DIR / "v10_trace.jsonl")
OPEN_TRACK_TRACE_PATH = str(TRACE_DIR / "v10_open_track_trace.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_v3_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1600,
    document_window_chars=3200,
)


## 2. Prompt

v10 沿用 v9 的保守回答与高阈值验证 prompt。主要优化点在工具读取覆盖率，而不是继续放宽 verifier。


In [ ]:
COMPACT_ANSWER_PROMPT = """
You are a conservative evidence-grounded QA agent for BrowseComp-Plus.
Use only the retrieved local corpus evidence in the conversation.

Rules:
1. Do not answer from memory.
2. If the evidence directly supports a short answer, provide it.
3. If evidence is weak, conflicting, or only loosely related, return 'Unable to determine' with low confidence.
4. Do not overfit to a familiar entity unless it satisfies the question's constraints.
5. For multi-hop questions, every hard clue must be directly supported before giving a concrete answer.
6. The exact answer must be a clean answer span, not a citation, bullet, document id, or evidence sentence.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()

PLANNER_PROMPT = """
Generate a compact BM25 search expansion plan. Do not answer the question.
Return JSON only with keys search_queries, key_phrases, keywords, answer_type.
Queries should be short, literal, and evidence-seeking.
""".strip()

DECOMPOSER_PROMPT = """
You are the planner agent in a multi-agent BrowseComp-Plus system.
Decompose the question into independently searchable evidence requirements.
Do not answer the question and do not use outside knowledge.
Return JSON only with keys:
- subquestions: short literal subquestions or clue checks
- constraints: dates, titles, roles, organizations, page ranges, quoted phrases, or other hard clues
- key_entities: names, places, organizations, works, or distinctive noun phrases
- expected_answer_type: one of person, title, date or year, percentage or number, organization, place, short answer
- search_focus: 2-4 concise BM25 query ideas that combine distinctive clues
""".strip()

ADJUDICATOR_PROMPT = """
You choose the safer final answer between two candidate answers for BrowseComp-Plus.
Use only the provided evidence summaries and candidate outputs.

Rules:
1. If one candidate is concrete and well-supported while the other is unsupported, choose the supported one.
2. If candidates conflict and neither is directly supported, choose 'Unable to determine'.
3. Do not invent a third answer unless it is explicitly stated in the evidence.
4. Output the required final answer format.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()

ANSWER_VERIFIER_PROMPT = """
You are the verifier agent for BrowseComp-Plus.
Use only the provided retrieved evidence and candidate answer.
Do not solve from memory.

Decide whether the candidate answer is directly supported by evidence and satisfies every question constraint. Mark unsupported if any key clue is missing, only topically related, or inferred from memory.
Return JSON only with keys:
- support_level: supported, unsupported, contradicted, or unclear
- supported: true or false
- answer: the clean exact answer span if supported, otherwise Unable to determine
- confidence: integer 0-100
- evidence: one short evidence summary
- missing_constraints: list of unsatisfied clues
""".strip()

OPEN_TRACK_FINALIZER_PROMPT = """
You are the finalizer agent in a planner-searcher-verifier BrowseComp-Plus system.
Use only retrieved local corpus evidence, including Open Track decomposition evidence.

Rules:
1. Provide a concrete answer only when one exact answer span is directly supported and matches the expected answer type.
2. The exact answer must be a clean span, not a bullet, document id, citation prefix, or quoted evidence fragment.
3. If a previous concrete candidate failed verification, do not repeat it unless new evidence explicitly satisfies every hard clue.
4. If evidence is partial, conflicting, or only topically related, return Unable to determine.
5. Do not use memory and do not infer beyond the retrieved evidence.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. 通用辅助函数

v10 继续保留 citation fragment salvage 和 malformed 过滤；新增的文档覆盖逻辑放在工具状态层。


In [ ]:
STOPWORDS = {
    "the", "and", "for", "that", "with", "from", "this", "what", "which", "who", "whose", "where",
    "when", "about", "into", "between", "after", "before", "during", "their", "there", "would",
    "could", "should", "please", "provide", "identify", "looking", "question", "answer", "according",
    "want", "know", "something", "simplicity", "refer", "them", "find", "documents", "matching", "clues",
}


def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def strip_thinking(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"(?is)<think>.*?</think>\s*", "", text).strip()
    text = re.sub(r"(?is)<think>.*", "", text).strip()
    return text


BAD_ANSWER_PREFIXES = (
    "explanation:", "evidence:", "confidence:", "based on", "the evidence", "from the evidence",
    "i cannot", "i can't", "there is not enough", "not enough evidence", "document ", "doc ",
    "citation", "source", "snippet", "quote:", "quoted evidence",
)

MALFORMED_ANSWER_PATTERNS = (
    r"^[-*•]\s*",
    r"(?i)^document\s+\d+\s*:",
    r"(?i)\bdocument\s+\d+\b",
    r"(?i)\bdocid\s*[:#]?\s*\d+\b",
    r"(?i)^quoted?\s+fragment\b",
)


def salvage_document_fragment_answer(raw_answer: str) -> str:
    text = re.sub(r"\s+", " ", str(raw_answer or "")).strip()
    match = re.search(r'(?i)document\s+\d+\s*:\s*["“]?([^"(;\n]{3,90})', text)
    if not match:
        return ""
    span = match.group(1)
    span = re.split(r"(?i)\b(?:annual report|form\s+10-k|quarterly report|report)\b", span)[0]
    span = re.sub(r"(?i)\b(?:incorporated|inc\.?|corp\.?|corporation|ltd\.?|limited|llc|plc)\b\.?", "", span)
    span = re.sub(r"\s+", " ", span).strip(" -–—:;,.\"'")
    if not span or len(span.split()) > 8:
        return ""
    return span


def is_malformed_answer(answer: str) -> bool:
    answer = str(answer or "").strip()
    if not answer:
        return True
    lower = answer.lower()
    if any(lower.startswith(prefix) for prefix in BAD_ANSWER_PREFIXES):
        return True
    if any(re.search(pattern, answer) for pattern in MALFORMED_ANSWER_PATTERNS):
        return True
    if answer.count('"') == 1 or answer.count("'") == 1:
        return True
    if len(answer) > 180 and len(answer.split()) > 24:
        return True
    if re.search(r"(?i)\b(could not|would not|was not|were not|had been|according to)\b", answer) and len(answer.split()) > 6:
        return True
    return False


def clean_answer_value(value: str) -> str:
    answer = strip_thinking(value)
    answer = re.split(r"(?im)^\s*(?:Explanation|Evidence|Confidence)\s*:", answer)[0].strip()
    raw_answer = re.sub(r"\s+", " ", answer).strip()
    salvaged = salvage_document_fragment_answer(raw_answer)
    if salvaged:
        return salvaged
    if is_malformed_answer(raw_answer):
        return "Unable to determine"
    answer = raw_answer.strip(" \t\n\r\"'`*_.,;:")
    if not answer:
        return "Unable to determine"
    lower = answer.lower()
    if any(lower.startswith(prefix) for prefix in BAD_ANSWER_PREFIXES):
        return "Unable to determine"
    if is_malformed_answer(answer):
        return "Unable to determine"
    return answer


def extract_exact_answer(text: str) -> str:
    original = str(text or "").strip()
    clean = strip_thinking(original)
    for candidate_text in (clean, original):
        match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return clean_answer_value(match.group(1))
        match = re.search(r"(?im)^\s*(?:Final\s+)?Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return clean_answer_value(match.group(1))
    lines = [line.strip() for line in clean.splitlines() if line.strip()]
    for line in reversed(lines):
        if re.match(r"(?i)^(explanation|evidence|confidence)\s*:", line):
            continue
        if 0 < len(line) <= 120:
            return clean_answer_value(line)
    return "Unable to determine"


def extract_confidence(text: str, default: int = 0) -> int:
    match = re.search(r"(?im)^\s*Confidence\s*:\s*(\d{1,3})\s*%", str(text or ""))
    if not match:
        return default
    return max(0, min(100, int(match.group(1))))


def coerce_confidence(value: Any, default: int = 0) -> int:
    try:
        return max(0, min(100, int(float(value))))
    except (TypeError, ValueError):
        return default


def is_unable_answer(answer: str) -> bool:
    return bool(re.search(r"(?i)unable|cannot|can't|not enough|insufficient|not determine|not specified|unknown", str(answer or "")))


def infer_expected_answer_type(question: str) -> str:
    q = canonical_text(question)
    if re.search(r"\b(percent|percentage|rate|ratio|decrease|increase)\b", q):
        return "percentage or number"
    if re.search(r"\b(when|date|year|month|day)\b", q):
        return "date or year"
    if re.search(r"\b(title|chapter|book|novel|song|album|film|movie|report|software|paper|article)\b", q):
        return "title"
    if re.search(r"\b(company|organization|organisation|university|ministry|agency|team|club|publisher)\b", q):
        return "organization"
    if re.search(r"\b(city|country|street|place|where|nationality)\b", q):
        return "place"
    if re.search(r"\b(who|person|name|author|designer|co-author|founder|director|actor|artist|professor)\b", q):
        return "person"
    return "short answer"


def should_expand(candidate: Dict[str, Any], min_confidence: int = 55) -> bool:
    answer = candidate.get("predicted_answer", "")
    confidence = int(candidate.get("confidence", 0) or 0)
    if is_unable_answer(answer):
        return True
    if is_malformed_answer(answer):
        return True
    if confidence and confidence < min_confidence:
        return True
    if len(str(answer).strip()) == 0 or len(str(answer)) > 180:
        return True
    return False


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


def normalize_string_list(value: Any, limit: int = 10, max_chars: int = 160) -> List[str]:
    if isinstance(value, str):
        items = [value]
    elif isinstance(value, list):
        items = value
    else:
        items = []
    output = []
    seen = set()
    for item in items:
        text = " ".join(str(item or "").replace("\n", " ").split()).strip(" ,.;:")
        if len(text) > max_chars:
            text = text[:max_chars].rstrip()
        key = text.lower()
        if text and key not in seen:
            output.append(text)
            seen.add(key)
        if len(output) >= limit:
            break
    return output


def tokenize_for_score(text: str) -> List[str]:
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-']+", str(text or "").lower())
    return [token.strip("'-") for token in tokens if len(token.strip("'-")) >= 4 and token not in STOPWORDS]


def rank_search_results(results: List[Dict[str, Any]], terms: List[str], limit: int = 8) -> List[Dict[str, Any]]:
    ranked = []
    seen_docids = set()
    for item in results:
        docid = str(item.get("docid", ""))
        if docid and docid in seen_docids:
            continue
        if docid:
            seen_docids.add(docid)
        haystack = canonical_text(" ".join([item.get("query", ""), item.get("snippet", ""), item.get("url", "")]))
        overlap = sum(1 for term in terms if term in haystack)
        score = float(item.get("score", 0) or 0)
        ranked.append({**item, "controller_score": overlap * 1000 + score, "term_overlap": overlap})
    ranked.sort(key=lambda row: (row.get("term_overlap", 0), row.get("score", 0)), reverse=True)
    return ranked[:limit]


def compact_result_summary(results: List[Dict[str, Any]], max_items: int = 8) -> List[Dict[str, Any]]:
    compact = []
    for item in results[:max_items]:
        compact.append(
            {
                "docid": item.get("docid", ""),
                "score": item.get("score", 0),
                "controller_score": item.get("controller_score", 0),
                "url": item.get("url", ""),
                "query": item.get("query", ""),
                "snippet": truncate_text(item.get("snippet", ""), 450),
            }
        )
    return compact


## 4. 工具执行与轨迹状态

v10 在 `state` 中记录 `opened_windows`，不再只记录 docid。这样同一文档可以在不同 offset 附近打开多个窗口，用来缓解“只读文档开头”的问题。


In [ ]:
def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4800) -> Any:
    if tool_name in {"get_document", "get_document_window"} and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in tool_registry:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[tool_name](**arguments)
        return {
            "ok": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": normalize_tool_result(tool_name, raw_result),
        }
    except Exception as exc:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": repr(exc)}


def make_tool_message(tool_call_id: str, executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {"role": "tool", "tool_call_id": tool_call_id, "content": json.dumps(content, ensure_ascii=False)}


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name in {"search", "search_many"} and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    return []


def compact_payload(value: Any, max_chars: int = 2200) -> Any:
    try:
        text = json.dumps(value, ensure_ascii=False, default=str)
    except TypeError:
        text = str(value)
    if len(text) <= max_chars:
        return value
    return {"_truncated_json": truncate_text(text, max_chars)}


def record_open_track_event(state: Dict[str, Any], component: str, action: str, payload: Dict[str, Any]) -> None:
    state.setdefault("open_track_trace", []).append(
        {
            "step": len(state.get("open_track_trace", [])) + 1,
            "component": component,
            "action": action,
            "payload": compact_payload(payload),
        }
    )


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "mode": "compact",
        "search_queries": [],
        "seen_docids": [],
        "opened_docids": [],
        "opened_windows": [],
        "ranked_results": [],
        "evidence_table": [],
        "tool_events": [],
        "candidate_answers": [],
        "focused_rescue_plan": {},
        "decomposition": {},
        "verification": {},
        "open_track_search": {},
        "open_track_trace": [],
        "decision": {},
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name in {"search", "search_many"}:
        return "docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:12])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} snippets"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> None:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    if tool_name == "search_many":
        for query in arguments.get("queries", []) or []:
            if query and query not in state["search_queries"]:
                state["search_queries"].append(query)
    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
    if tool_name == "get_document_window" and isinstance(result, dict):
        window_record = {
            "docid": str(result.get("docid", "")),
            "start": int(result.get("start", 0) or 0),
            "end": int(result.get("end", 0) or 0),
            "text_length": int(result.get("text_length", 0) or 0),
        }
        if window_record["docid"] and window_record not in state.setdefault("opened_windows", []):
            state["opened_windows"].append(window_record)
    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "summary": summarize_tool_result(tool_name, result)}
        )
    state["tool_events"].append(
        {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "ok": executed.get("ok", False), "summary": summarize_tool_result(tool_name, result)}
    )


def append_controller_tool_call(messages: List[Dict[str, Any]], state: Dict[str, Any], tool_name: str, arguments: Dict[str, Any], round_id: int, note: str) -> Dict[str, Any]:
    call_id = f"controller_{round_id}_{len(state['tool_events']) + 1}_{tool_name}"
    messages.append(
        {
            "role": "assistant",
            "content": note,
            "tool_calls": [
                {"id": call_id, "type": "function", "function": {"name": tool_name, "arguments": json.dumps(arguments, ensure_ascii=False)}}
            ],
        }
    )
    executed = execute_tool(tool_name, arguments)
    messages.append(make_tool_message(call_id, executed))
    update_state_from_execution(state, executed, round_id)
    return executed


def window_already_opened(state: Dict[str, Any], docid: str, start: int, tolerance: int = 900) -> bool:
    for window in state.get("opened_windows", []):
        if str(window.get("docid", "")) != str(docid):
            continue
        if abs(int(window.get("start", 0) or 0) - int(start or 0)) <= tolerance:
            return True
    return False


def open_context_windows_from_snippets(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    snippet_result: Any,
    round_id: int,
    max_windows: int = 2,
    window_chars: int = 3600,
    note: str = "v10 evidence coverage: inspect keyword-hit context window.",
) -> List[Dict[str, Any]]:
    if not isinstance(snippet_result, dict):
        return []
    opened = []
    seen = set()
    snippets = snippet_result.get("snippets", []) or []
    snippets = sorted(
        [item for item in snippets if item.get("docid") and item.get("start") is not None],
        key=lambda item: (str(item.get("docid", "")), int(item.get("start", 0) or 0)),
    )
    for item in snippets:
        docid = str(item.get("docid", ""))
        start = max(0, int(item.get("start", 0) or 0) - window_chars // 3)
        key = (docid, start // 900)
        if key in seen or window_already_opened(state, docid, start):
            continue
        seen.add(key)
        executed = append_controller_tool_call(
            messages,
            state,
            "get_document_window",
            {"docid": docid, "start": start, "window_chars": window_chars},
            round_id=round_id,
            note=note,
        )
        opened.append(executed)
        if len(opened) >= max_windows:
            break
    return opened


MODEL_INPUT_BYTE_BUDGET = 26000
MAX_STATE_MESSAGE_BYTES = 8000
MAX_TOOL_SNIPPET_BYTES = 900
MAX_TOOL_RESULT_BYTES = 1800
MAX_ASSISTANT_CONTEXT_BYTES = 800
MAX_MODEL_MESSAGES = 28


def truncate_to_bytes(text: str, max_bytes: int) -> str:
    text = str(text or "")
    encoded = text.encode("utf-8")
    if len(encoded) <= max_bytes:
        return text
    suffix = "... [byte-truncated]"
    suffix_bytes = suffix.encode("utf-8")
    keep = max(0, int(max_bytes) - len(suffix_bytes))
    return encoded[:keep].decode("utf-8", errors="ignore") + suffix


def compact_tool_payload_for_prompt(value: Any) -> Any:
    if isinstance(value, list):
        return [compact_tool_payload_for_prompt(item) for item in value[:8]]
    if not isinstance(value, dict):
        if isinstance(value, str):
            return truncate_to_bytes(value, MAX_TOOL_RESULT_BYTES)
        return value
    compact: Dict[str, Any] = {}
    for key, item in value.items():
        if key in {"snippet", "text"}:
            compact[key] = truncate_to_bytes(str(item or ""), MAX_TOOL_SNIPPET_BYTES)
        elif key in {"snippets", "matches"} and isinstance(item, list):
            compact[key] = [compact_tool_payload_for_prompt(part) for part in item[:8]]
        elif key == "raw_content":
            compact[key] = truncate_to_bytes(str(item or ""), 350)
        elif isinstance(item, (dict, list)):
            compact[key] = compact_tool_payload_for_prompt(item)
        elif isinstance(item, str):
            compact[key] = truncate_to_bytes(item, 350)
        else:
            compact[key] = item
    return compact


def compact_message_for_model(message: Dict[str, Any]) -> Dict[str, Any]:
    role = message.get("role", "user")
    content = message.get("content", "")
    if role == "tool":
        try:
            payload = json.loads(content)
            content = json.dumps(compact_tool_payload_for_prompt(payload), ensure_ascii=False)
        except Exception:
            content = truncate_to_bytes(str(content or ""), MAX_TOOL_RESULT_BYTES)
        return {"role": "user", "content": "Tool result:\n" + truncate_to_bytes(content, MAX_TOOL_RESULT_BYTES)}
    if "tool_calls" in message:
        calls = []
        for call in message.get("tool_calls", []) or []:
            function = call.get("function", {})
            calls.append({"name": function.get("name", ""), "arguments": function.get("arguments", "")})
        content = f"{content}\nController tool call summary:\n{json.dumps(calls, ensure_ascii=False)}"
        return {"role": "assistant", "content": truncate_to_bytes(content, MAX_ASSISTANT_CONTEXT_BYTES)}
    if isinstance(content, str):
        if content.startswith("Compressed evidence state:"):
            content = truncate_to_bytes(content, MAX_STATE_MESSAGE_BYTES)
        elif role == "assistant":
            content = truncate_to_bytes(content, MAX_ASSISTANT_CONTEXT_BYTES)
        else:
            content = truncate_to_bytes(content, 3000)
    return {"role": role, "content": content}


def total_message_bytes(messages: List[Dict[str, Any]]) -> int:
    return len(json.dumps(messages, ensure_ascii=False).encode("utf-8"))


def compact_messages_for_model(messages: List[Dict[str, Any]], byte_budget: int = MODEL_INPUT_BYTE_BUDGET) -> List[Dict[str, Any]]:
    compacted = [compact_message_for_model(message) for message in messages]
    while len(compacted) > MAX_MODEL_MESSAGES:
        compacted.pop(3)
    while total_message_bytes(compacted) > byte_budget and len(compacted) > 5:
        compacted.pop(3)
    if total_message_bytes(compacted) > byte_budget and compacted:
        per_message = max(500, byte_budget // max(len(compacted), 1))
        for index, message in enumerate(compacted):
            if index == 0:
                limit = min(3500, per_message)
            elif index == 1:
                limit = min(5000, per_message)
            else:
                limit = per_message
            message["content"] = truncate_to_bytes(str(message.get("content", "")), limit)
    while total_message_bytes(compacted) > byte_budget and len(compacted) > 3:
        compacted.pop(-2)
    return compacted


def make_state_message(state: Dict[str, Any], max_bytes: int = MAX_STATE_MESSAGE_BYTES) -> Dict[str, Any]:
    content = "Compressed evidence state:\n" + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
    return {"role": "user", "content": truncate_to_bytes(content, max_bytes)}


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "mode": state["mode"],
        "search_queries": state["search_queries"][-10:],
        "seen_docids": state["seen_docids"][-30:],
        "opened_docids": state["opened_docids"][-14:],
        "opened_windows": state.get("opened_windows", [])[-12:],
        "ranked_results": compact_result_summary(state["ranked_results"], max_items=8),
        "evidence_table": state["evidence_table"][-8:],
        "candidate_answers": state["candidate_answers"],
        "focused_rescue_plan": state.get("focused_rescue_plan", {}),
        "open_track": {
            "decomposition": state.get("decomposition", {}),
            "verification": state.get("verification", {}),
            "search": state.get("open_track_search", {}),
            "trace": state.get("open_track_trace", [])[-8:],
        },
        "decision": state["decision"],
    }


## 5. Compact Path 与 Expansion Path

主干仍是 compact -> expansion -> adjudication。v10 在 expansion 的 evidence snippets 后追加打开命中位置上下文窗口，再交给模型回答。


In [ ]:
def answer_from_state(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], prompt: str, max_tokens: int = 800) -> Dict[str, Any]:
    state_message = make_state_message(state)
    final_instruction = {"role": "user", "content": prompt}
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=compact_messages_for_model([messages[0], messages[1], state_message] + messages[2:] + [final_instruction]),
        temperature=0.0,
        max_tokens=max_tokens,
    )
    content = response["choices"][0]["message"].get("content", "")
    return {"content": content, "predicted_answer": extract_exact_answer(content), "confidence": extract_confidence(content)}


def run_compact_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 2) -> Dict[str, Any]:
    state["mode"] = "compact"
    queries = make_initial_search_queries(question, max_queries=4, max_query_chars=220)
    if not queries:
        queries = [question]
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": queries, "per_query_k": per_query_k},
        round_id=0,
        note="v10 compact path: initial BM25 searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question)
    ranked = rank_search_results(raw_results, terms=terms, limit=8)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "v10 compact ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    for item in ranked[:auto_open_docs]:
        docid = str(item.get("docid", ""))
        if docid:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 2800},
                round_id=0,
                note="v10 compact path: inspect compact candidate.",
            )
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "compact"
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate


def plan_expansion(question: str) -> Dict[str, Any]:
    fallback = {"search_queries": make_initial_search_queries(question, max_queries=5, max_query_chars=180), "key_phrases": [], "keywords": tokenize_for_score(question)[:16], "answer_type": "short answer"}
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": PLANNER_PROMPT}, {"role": "user", "content": question}],
            temperature=0.0,
            max_tokens=500,
        )
        parsed = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback
    planner_queries = normalize_string_list(parsed.get("search_queries"), limit=7, max_chars=150)
    deterministic = make_initial_search_queries(question, max_queries=4, max_query_chars=180)
    key_phrases = normalize_string_list(parsed.get("key_phrases"), limit=10, max_chars=100)
    keywords = normalize_string_list(parsed.get("keywords"), limit=14, max_chars=60)
    return {
        "search_queries": normalize_string_list(deterministic + planner_queries + key_phrases, limit=8, max_chars=150),
        "key_phrases": key_phrases,
        "keywords": keywords,
        "answer_type": str(parsed.get("answer_type", "short answer")),
    }


def run_expansion_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 3) -> Dict[str, Any]:
    state["mode"] = "expanded"
    plan = plan_expansion(question)
    messages.append({"role": "assistant", "content": "v10 expansion planner:\n" + json.dumps(plan, ensure_ascii=False, indent=2)})
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": plan["search_queries"], "per_query_k": per_query_k},
        round_id=1,
        note="v10 expansion path: execute additional planned searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(" ".join(plan.get("keywords", []) + plan.get("key_phrases", [])))
    ranked = rank_search_results(raw_results, terms=terms, limit=10)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "v10 expansion ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    opened = 0
    for item in ranked:
        docid = str(item.get("docid", ""))
        if docid and docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 2800},
                round_id=1,
                note="v10 expansion path: inspect additional candidate.",
            )
            opened += 1
        if opened >= auto_open_docs:
            break
    keywords = ", ".join(normalize_string_list(plan.get("keywords") + plan.get("key_phrases"), limit=12, max_chars=80))
    top_docids = [str(item.get("docid", "")) for item in ranked[:8] if item.get("docid")]
    if keywords and top_docids:
        snippet_execution = append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": keywords, "window_chars": 1000, "max_snippets": 12},
            round_id=1,
            note="v10 expansion path: collect additional evidence snippets.",
        )
        open_context_windows_from_snippets(
            messages,
            state,
            snippet_execution.get("tool_result"),
            round_id=1,
            max_windows=1,
            window_chars=2800,
            note="v10 expansion path: inspect evidence-hit context window.",
        )
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "expanded"
    candidate["plan"] = plan
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate


## 6. Open Track 工具、Agent 架构与 v10 控制器

v10 保留 v9 的高阈值 verifier。Open Track searcher 在收集 decomposition evidence snippets 后，会额外打开 3 个非重复的命中位置窗口，优先补足长文档中部/后部证据。


In [ ]:
def adjudicate_candidates(question: str, state: Dict[str, Any], compact: Dict[str, Any], expanded: Dict[str, Any]) -> Dict[str, Any]:
    if not should_expand(compact):
        return {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}
    if should_expand(compact) and not should_expand(expanded):
        return {**expanded, "selected_mode": "expanded", "decision_reason": "compact_weak_expanded_concrete"}
    if is_unable_answer(compact.get("predicted_answer", "")) and is_unable_answer(expanded.get("predicted_answer", "")):
        return {**compact, "selected_mode": "compact", "decision_reason": "both_unable_keep_conservative"}

    evidence_summary = truncate_text(json.dumps(public_state_summary(state), ensure_ascii=False, indent=2), MAX_STATE_MESSAGE_BYTES)
    adjudication_input = {
        "role": "user",
        "content": (
            f"Question:\n{question}\n\n"
            f"Compact candidate:\n{compact.get('content', '')}\n\n"
            f"Expanded candidate:\n{expanded.get('content', '')}\n\n"
            f"Evidence summary:\n{evidence_summary}"
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": ADJUDICATOR_PROMPT}, adjudication_input],
            temperature=0.0,
            max_tokens=500,
        )
        content = response["choices"][0]["message"].get("content", "")
        return {
            "mode": "adjudicated",
            "selected_mode": "adjudicated",
            "decision_reason": "adjudicator_used_for_conflict_or_weak_answers",
            "content": content,
            "predicted_answer": extract_exact_answer(content),
            "confidence": extract_confidence(content),
        }
    except Exception:
        if not is_unable_answer(expanded.get("predicted_answer", "")):
            return {**expanded, "selected_mode": "expanded", "decision_reason": "adjudicator_failed_keep_expanded"}
        return {**compact, "selected_mode": "compact", "decision_reason": "adjudicator_failed_keep_compact"}


def deterministic_decomposition(question: str) -> Dict[str, Any]:
    expected_type = infer_expected_answer_type(question)
    quoted = re.findall(r"[\"'“”‘’]([^\"'“”‘’]{3,90})[\"'“”‘’]", question)
    years = re.findall(r"\b(?:1[5-9]\d{2}|20\d{2})\b", question)
    terms = tokenize_for_score(question)
    distinctive = normalize_string_list(quoted + years + terms, limit=12, max_chars=80)
    clue_text = " ".join(distinctive[:10]) or question
    subquestions = normalize_string_list(
        [
            f"Find documents matching these clues: {clue_text}",
            f"Identify the {expected_type} requested by the question",
            f"Verify all hard constraints before answering: {' '.join(distinctive[:8])}",
        ],
        limit=5,
        max_chars=170,
    )
    search_focus = normalize_string_list(
        [clue_text, f"{clue_text} {expected_type}", f"{expected_type} {' '.join(distinctive[:6])}"],
        limit=4,
        max_chars=180,
    )
    return {
        "subquestions": subquestions,
        "constraints": distinctive,
        "key_entities": distinctive[:8],
        "expected_answer_type": expected_type,
        "search_focus": search_focus,
    }


def decompose_question_for_open_track(question: str, state: Dict[str, Any]) -> Dict[str, Any]:
    fallback = deterministic_decomposition(question)
    planner_input = {
        "role": "user",
        "content": json.dumps(
            {
                "question": question,
                "expected_answer_type": fallback["expected_answer_type"],
                "previous_search_queries": state.get("search_queries", [])[-10:],
                "opened_docids": state.get("opened_docids", [])[-10:],
                "top_ranked_results": compact_result_summary(state.get("ranked_results", []), max_items=6),
            },
            ensure_ascii=False,
            indent=2,
        ),
    }
    source = "model"
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": DECOMPOSER_PROMPT}, planner_input],
            temperature=0.0,
            max_tokens=500,
        )
        parsed = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback
        source = "deterministic_fallback"
    decomposition = {
        "subquestions": normalize_string_list(parsed.get("subquestions") or fallback["subquestions"], limit=6, max_chars=170),
        "constraints": normalize_string_list(parsed.get("constraints") or fallback["constraints"], limit=14, max_chars=90),
        "key_entities": normalize_string_list(parsed.get("key_entities") or fallback["key_entities"], limit=10, max_chars=90),
        "expected_answer_type": str(parsed.get("expected_answer_type") or parsed.get("answer_type") or fallback["expected_answer_type"]),
        "search_focus": normalize_string_list(parsed.get("search_focus") or fallback["search_focus"], limit=5, max_chars=180),
        "source": source,
    }
    state["decomposition"] = decomposition
    record_open_track_event(state, "planner_agent", "decompose_question", decomposition)
    return decomposition


def build_decomposition_queries(question: str, decomposition: Dict[str, Any], limit: int = 6) -> List[str]:
    queries: List[str] = []
    entities = decomposition.get("key_entities", [])
    constraints = decomposition.get("constraints", [])
    answer_type = str(decomposition.get("expected_answer_type", "short answer"))
    queries.extend(decomposition.get("search_focus", [])[:4])
    if entities and constraints:
        queries.append(" ".join((entities[:5] + constraints[:5])[:10]))
    if constraints:
        queries.append(" ".join(constraints[:10]))
        queries.append(f"{answer_type} {' '.join(constraints[:8])}")
    queries.extend(make_initial_search_queries(question, max_queries=3, max_query_chars=180))
    cleaned = []
    for query in normalize_string_list(queries, limit=limit + 4, max_chars=180):
        q = canonical_text(query)
        if q.startswith(("find documents matching", "identify the", "verify all hard constraints")):
            continue
        if len(tokenize_for_score(query)) < 3:
            continue
        cleaned.append(query)
        if len(cleaned) >= limit:
            break
    return cleaned or make_initial_search_queries(question, max_queries=3, max_query_chars=180) or [question]


def run_open_track_searcher_agent(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    decomposition: Dict[str, Any],
    per_query_k: int = 4,
    auto_open_docs: int = 3,
) -> Dict[str, Any]:
    state["mode"] = "open_track_search"
    planned_queries = build_decomposition_queries(question, decomposition, limit=7)
    previous_queries = {canonical_text(query) for query in state.get("search_queries", [])}
    queries = [query for query in planned_queries if canonical_text(query) not in previous_queries]
    search_state = {"queries": queries, "opened_docids": [], "top_docids": [], "keywords": []}
    state["open_track_search"] = search_state
    if not queries:
        record_open_track_event(state, "searcher_agent", "search_skipped", {"reason": "all decomposition queries duplicate previous queries"})
        return search_state

    messages.append({"role": "assistant", "content": "v10 open-track searcher plan:\n" + json.dumps({"queries": queries, "decomposition": decomposition}, ensure_ascii=False, indent=2)})
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": queries, "per_query_k": per_query_k},
        round_id=2,
        note="v10 open-track searcher: execute decomposition queries.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(
        " ".join(decomposition.get("constraints", []) + decomposition.get("key_entities", []) + [decomposition.get("expected_answer_type", "")])
    )
    ranked = rank_search_results(raw_results, terms=terms, limit=10)
    state["ranked_results"] = rank_search_results(state.get("ranked_results", []) + ranked, terms=terms, limit=12)
    messages.append({"role": "assistant", "content": "v10 open-track ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})

    opened_docids: List[str] = []
    for item in ranked:
        docid = str(item.get("docid", ""))
        if docid and docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 3000},
                round_id=2,
                note="v10 open-track searcher: inspect decomposition candidate.",
            )
            opened_docids.append(docid)
        if len(opened_docids) >= auto_open_docs:
            break

    keywords = normalize_string_list(
        decomposition.get("constraints", []) + decomposition.get("key_entities", []) + tokenize_for_score(decomposition.get("expected_answer_type", "")),
        limit=14,
        max_chars=80,
    )
    top_docids = [str(item.get("docid", "")) for item in ranked[:8] if item.get("docid")]
    if keywords and top_docids:
        snippet_execution = append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": ", ".join(keywords), "window_chars": 1000, "max_snippets": 12},
            round_id=2,
            note="v10 open-track searcher: collect decomposition evidence snippets.",
        )
        opened_contexts = open_context_windows_from_snippets(
            messages,
            state,
            snippet_execution.get("tool_result"),
            round_id=2,
            max_windows=2,
            window_chars=3000,
            note="v10 open-track searcher: inspect decomposition evidence-hit context window.",
        )

    search_state.update(
        {
            "opened_docids": opened_docids,
            "top_docids": top_docids,
            "keywords": keywords,
            "retrieved_count": len(raw_results),
            "context_windows_opened": len(opened_contexts) if "opened_contexts" in locals() else 0,
        }
    )
    state["open_track_search"] = search_state
    record_open_track_event(state, "searcher_agent", "decomposition_search", search_state)
    return search_state


def normalize_verification(parsed: Dict[str, Any], candidate_answer: str, raw_content: str = "") -> Dict[str, Any]:
    support_level = canonical_text(parsed.get("support_level", "unclear")) or "unclear"
    if support_level in {"not supported", "not_supported", "no support", "unsupported answer"}:
        support_level = "unsupported"
    elif "contradict" in support_level:
        support_level = "contradicted"
    elif support_level in {"directly supported", "fully supported", "support"}:
        support_level = "supported"
    supported_value = parsed.get("supported")
    if isinstance(supported_value, str):
        supported = supported_value.strip().lower() in {"true", "yes", "supported"}
    elif supported_value is None:
        supported = support_level in {"supported", "directly supported", "fully supported"}
    else:
        supported = bool(supported_value)
    verified_answer = clean_answer_value(parsed.get("answer") or candidate_answer)
    confidence = coerce_confidence(parsed.get("confidence"), default=0)
    if is_unable_answer(verified_answer) or is_malformed_answer(verified_answer):
        supported = False
    return {
        "support_level": support_level,
        "supported": supported,
        "answer": verified_answer,
        "confidence": confidence,
        "evidence": truncate_text(str(parsed.get("evidence", "")), 600),
        "missing_constraints": normalize_string_list(parsed.get("missing_constraints"), limit=8, max_chars=100),
        "raw_content": truncate_text(raw_content, 900),
    }


def verify_candidate_answer(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    candidate_answer = clean_answer_value(candidate.get("predicted_answer", ""))
    if is_unable_answer(candidate_answer):
        verification = {
            "support_level": "not_applicable",
            "supported": False,
            "answer": "Unable to determine",
            "confidence": 0,
            "evidence": "candidate is Unable to determine",
            "missing_constraints": [],
            "raw_content": "",
        }
        state["verification"] = verification
        record_open_track_event(state, "verifier_agent", "verify_claim", {"candidate_answer": candidate_answer, "verification": verification})
        return verification

    state_message = make_state_message(state)
    verifier_input = {
        "role": "user",
        "content": (
            f"Question:\n{question}\n\n"
            f"Candidate answer:\n{candidate_answer}\n\n"
            f"Candidate output:\n{candidate.get('content', '')}\n\n"
            "Verify whether the candidate answer is directly supported by the retrieved evidence."
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=compact_messages_for_model([{"role": "system", "content": ANSWER_VERIFIER_PROMPT}, messages[1], state_message] + messages[2:] + [verifier_input]),
            temperature=0.0,
            max_tokens=500,
        )
        content = response["choices"][0]["message"].get("content", "")
        parsed = extract_json_object(content) or {}
        verification = normalize_verification(parsed, candidate_answer=candidate_answer, raw_content=content)
    except Exception as exc:
        verification = {
            "support_level": "verifier_error",
            "supported": False,
            "answer": candidate_answer,
            "confidence": 0,
            "evidence": f"verifier failed: {exc}",
            "missing_constraints": [],
            "raw_content": "",
        }
    state["verification"] = verification
    record_open_track_event(state, "verifier_agent", "verify_claim", {"candidate_answer": candidate_answer, "verification": verification})
    return verification


VERIFIED_ACCEPT_CONFIDENCE = 95
OPEN_TRACK_RESCUE_CONFIDENCE = 95


def verification_has_missing_constraints(verification: Dict[str, Any]) -> bool:
    return bool(verification.get("missing_constraints"))


def verification_accepts(verification: Dict[str, Any], min_confidence: int = VERIFIED_ACCEPT_CONFIDENCE) -> bool:
    return (
        bool(verification.get("supported"))
        and coerce_confidence(verification.get("confidence"), default=0) >= min_confidence
        and not verification_has_missing_constraints(verification)
        and not is_unable_answer(verification.get("answer", ""))
        and not is_malformed_answer(verification.get("answer", ""))
    )


def verification_is_weak_supported(verification: Optional[Dict[str, Any]]) -> bool:
    if not verification:
        return False
    return bool(verification.get("supported")) and not verification_accepts(verification)


def open_track_finalize_answer(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    compact: Dict[str, Any],
    expanded: Optional[Dict[str, Any]],
    selected: Dict[str, Any],
) -> Dict[str, Any]:
    expected_type = state.get("decomposition", {}).get("expected_answer_type") or infer_expected_answer_type(question)
    state_message = make_state_message(state)
    finalizer_input = {
        "role": "user",
        "content": (
            f"Question:\n{question}\n\n"
            f"Expected answer type: {expected_type}\n\n"
            f"Compact candidate:\n{compact.get('content', '')}\n\n"
            f"Expanded candidate:\n{expanded.get('content', '') if expanded else ''}\n\n"
            f"Currently selected candidate:\n{selected.get('content', '')}\n\n"
            "If one exact answer is now directly extractable from the tool evidence, answer it. Otherwise keep Unable to determine."
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=compact_messages_for_model([{"role": "system", "content": OPEN_TRACK_FINALIZER_PROMPT}, messages[1], state_message] + messages[2:] + [finalizer_input]),
            temperature=0.0,
            max_tokens=500,
        )
        content = response["choices"][0]["message"].get("content", "")
    except Exception as exc:
        content = f"Explanation: Open Track finalizer failed: {exc}\nEvidence: none\nExact Answer: Unable to determine\nConfidence: 0%"
    candidate = {
        "mode": "open_track_finalized",
        "selected_mode": "open_track_finalized",
        "decision_reason": "open_track_planner_searcher_finalizer",
        "content": content,
        "predicted_answer": extract_exact_answer(content),
        "confidence": extract_confidence(content),
        "expected_answer_type": expected_type,
    }
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    record_open_track_event(state, "finalizer_agent", "finalize_answer", {k: candidate[k] for k in ("mode", "predicted_answer", "confidence", "expected_answer_type")})
    return candidate


def make_unable_selection(reason: str, source: Dict[str, Any]) -> Dict[str, Any]:
    content = f"Explanation: {reason}\nEvidence: verification did not directly support a clean answer span\nExact Answer: Unable to determine\nConfidence: 0%"
    return {
        "mode": "open_track_rejected",
        "selected_mode": "open_track_rejected",
        "decision_reason": reason,
        "content": content,
        "predicted_answer": "Unable to determine",
        "confidence": 0,
        "rejected_candidate": source.get("predicted_answer", ""),
    }


def should_attempt_open_track_search(state: Dict[str, Any], expanded: Optional[Dict[str, Any]], selected: Dict[str, Any], verification: Optional[Dict[str, Any]] = None) -> bool:
    if expanded is None:
        return False
    answer = selected.get("predicted_answer", "")
    if is_unable_answer(answer) or is_malformed_answer(answer):
        return True
    if verification and verification.get("support_level") in {"unsupported", "contradicted"}:
        return True
    if coerce_confidence(selected.get("confidence"), default=0) < 50:
        return True
    return False


def apply_open_track_agents(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    compact: Dict[str, Any],
    expanded: Optional[Dict[str, Any]],
    selected: Dict[str, Any],
) -> Dict[str, Any]:
    selected_answer = selected.get("predicted_answer", "")
    selected_verification: Optional[Dict[str, Any]] = None

    if not is_unable_answer(selected_answer):
        selected_verification = verify_candidate_answer(question, messages, state, selected)
        if verification_accepts(selected_verification):
            return {
                **selected,
                "predicted_answer": selected_verification.get("answer", selected.get("predicted_answer", "")),
                "decision_reason": selected.get("decision_reason", "") + "_verified_high_confidence",
            }

        if expanded is None:
            if verification_is_weak_supported(selected_verification):
                return make_unable_selection("open_track_rejected_weak_supported_compact", selected)
            if is_malformed_answer(selected_answer):
                return make_unable_selection("open_track_rejected_malformed_answer", selected)
            return {**selected, "decision_reason": selected.get("decision_reason", "") + "_verification_nonblocking_kept"}

    if should_attempt_open_track_search(state, expanded, selected, selected_verification):
        decomposition = decompose_question_for_open_track(question, state)
        run_open_track_searcher_agent(question, messages, state, decomposition)
        open_track_candidate = open_track_finalize_answer(question, messages, state, compact, expanded, selected)
        if not is_unable_answer(open_track_candidate.get("predicted_answer", "")):
            open_track_verification = verify_candidate_answer(question, messages, state, open_track_candidate)
            if verification_accepts(open_track_verification, min_confidence=OPEN_TRACK_RESCUE_CONFIDENCE):
                return {
                    **open_track_candidate,
                    "predicted_answer": open_track_verification.get("answer", open_track_candidate.get("predicted_answer", "")),
                    "decision_reason": "open_track_verified_rescue_high_confidence",
                }
        if selected_verification and not verification_accepts(selected_verification):
            return make_unable_selection("open_track_rejected_low_confidence_or_incomplete_concrete", selected)
        return {**selected, "decision_reason": selected.get("decision_reason", "") + "_open_track_kept_original"}

    if is_malformed_answer(selected_answer):
        return make_unable_selection("open_track_rejected_malformed_answer", selected)
    return selected


def build_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "candidate_answers": state.get("candidate_answers", []),
        "search_queries": state.get("search_queries", []),
        "seen_docids": state.get("seen_docids", []),
        "opened_docids": state.get("opened_docids", []),
        "opened_windows": state.get("opened_windows", []),
        "evidence_table": state.get("evidence_table", []),
        "ranked_results": state.get("ranked_results", []),
        "focused_rescue_plan": state.get("focused_rescue_plan", {}),
        "open_track": state.get("open_track", {}),
    }


def build_open_track_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    open_track = state.get("open_track", {})
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "decomposition": open_track.get("decomposition", {}),
        "verification": open_track.get("verification", {}),
        "search": open_track.get("search", {}),
        "events": open_track.get("trace", []),
    }


def run_v10_agent(
    question: str,
    query_id: Optional[str] = None,
    compact_per_query_k: int = 5,
    compact_auto_open_docs: int = 2,
    expansion_per_query_k: int = 5,
    expansion_auto_open_docs: int = 3,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": COMPACT_ANSWER_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    compact = run_compact_path(question, messages, state, per_query_k=compact_per_query_k, auto_open_docs=compact_auto_open_docs)

    expanded = None
    if should_expand(compact):
        expanded = run_expansion_path(question, messages, state, per_query_k=expansion_per_query_k, auto_open_docs=expansion_auto_open_docs)
        selected = adjudicate_candidates(question, state, compact, expanded)
    else:
        selected = {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}

    selected = apply_open_track_agents(question, messages, state, compact, expanded, selected)

    state["decision"] = {
        "selected_mode": selected.get("selected_mode", selected.get("mode", "unknown")),
        "decision_reason": selected.get("decision_reason", ""),
        "compact_predicted_answer": compact.get("predicted_answer", ""),
        "compact_confidence": compact.get("confidence", 0),
        "expanded_predicted_answer": expanded.get("predicted_answer", "") if expanded else None,
        "expanded_confidence": expanded.get("confidence", 0) if expanded else None,
        "open_track_event_count": len(state.get("open_track_trace", [])),
        "open_track_verification": state.get("verification", {}),
        "open_track_search": state.get("open_track_search", {}),
    }

    messages.append({"role": "assistant", "content": selected.get("content", "")})
    return {
        "query_id": query_id,
        "status": "completed_v10_evidence_window_coverage",
        "predicted_answer": selected.get("predicted_answer", ""),
        "final_output": selected.get("content", ""),
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 7. Submission、Trace 与评测

`generate_submission()` 同时写 submission、主 trace 和 Open Track 组件 trace。v10 的 trace 新增 `opened_windows`，用于判断非开头窗口是否真的被打开。


In [ ]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    result = run_v10_agent(question=row["query"], query_id=str(row.get("query_id", "")), **agent_kwargs)
    record = {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }
    trace = build_trace_record(row, result)
    open_track_trace = build_open_track_trace_record(row, result)
    return record, trace, open_track_trace


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    trace_path: str = TRACE_PATH,
    open_track_trace_path: str = OPEN_TRACK_TRACE_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    trace_file = Path(trace_path)
    open_track_trace_file = Path(open_track_trace_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    trace_file.parent.mkdir(parents=True, exist_ok=True)
    open_track_trace_file.parent.mkdir(parents=True, exist_ok=True)
    records = []
    with output_file.open("w", encoding="utf-8") as fout, trace_file.open("w", encoding="utf-8") as trout, open_track_trace_file.open("w", encoding="utf-8") as otout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record, trace, open_track_trace = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            trout.write(json.dumps(trace, ensure_ascii=False) + "\n")
            otout.write(json.dumps(open_track_trace, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {output_path}")
    print(f"Saved trace records to {trace_path}")
    print(f"Saved Open Track trace records to {open_track_trace_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        max_tokens=1024,
        verbose=True,
    )


## 8. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50)
# summary, details = evaluate_submission()
```


In [ ]:
# records = generate_submission(limit=50)
# summary, details = evaluate_submission()
